# 第 12 周 · 周二：针对性补漏（上）——最常薄弱的五大技术模块

> 难度：⭐⭐⭐⭐ | 定位：针对窗口函数、AB实验、归因模型、因果推断、SQL优化这五个最常薄弱的模块做深度强化
> 前置依赖：W1-W11全部内容 + 昨天的盲区排查结果 | 后续用途：周四综合模拟、面试准备
> 面试地位：这五个模块是大厂面试的"死亡区"——大多数人在这些模块上被追问到露馅

---

## 📘 区块1：核心概念与第一性原理

### 1.1 为什么需要"针对性补漏"？

学完11周，你不是"所有模块都强"。更真实的情况是：
- 有的模块闭眼能做（比如指标拆解、漏斗分析）
- 有的模块"知道有这事但做不出来"（比如gap-and-island、MDE计算）
- 有的模块"学的时候觉得懂了，用的时候发现完全不会"（比如DID的前提假设检验）

**针对性补漏的本质**：不是"重新学一遍"，而是找到每个模块最核心的卡点，用刻意练习把它打通。

### 1.2 从第一性原理推导：为什么卡在"从会到能"？

学习的三个阶段：
1. **理解**：知道这个工具/方法是什么（阅读/听课即可达到）
2. **模仿**：看着例子能复现（跟着教程做即可达到）
3. **应用**：面对新问题能独立选择并正确使用工具（需要大量刻意练习）

大多数人的学习卡在第2和第3阶段之间——因为课程只能带你到"模仿"，而"应用"需要你自己在练习中完成。这就是今天要解决的问题。

### 1.3 五大模块的常见卡点一览

| 模块 | 最常见的卡点 | 根因 | 突破方法 |
|------|------------|------|----------|
| 窗口函数 | 知道语法但面对新需求时不知道该用哪个函数 | 没有建立"需求→函数"的映射 | 背熟5类需求→函数的映射表 |
| AB实验 | 知道p<0.05但不会算样本量、不会识别Peeking | 只学了概念没学实操 | 手算一遍完整的样本量+审阅一份有问题的报告 |
| 归因模型 | 只会末次点击，不知道什么场景换什么模型 | 没理解各模型的假设前提 | 同一份数据用3种模型算，看差异 |
| 因果推断 | 知道DID/PSM/IV/RDD的名字但说不清前提假设 | 只记了结论没理解逻辑 | 对每种方法列出前提假设+检验方法+违反后果 |
| SQL优化 | 知道要加索引但不知道加什么、怎么验证 | 不会用EXPLAIN分析执行计划 | 拿慢查询实际分析一遍EXPLAIN |

### 1.4 补漏的最佳方法论：刻意练习四步法

**第一步：找到具体卡点**
不是"我SQL不行"，而是"我写不出gap-and-island的SQL"。越具体越好。

**第二步：获取正确的解题模式**
找到参考答案或正确做法，理解每一步为什么这么做——不是"背下来"，是"理解为什么"。

**第三步：变式练习**
在原始题目的基础上改参数、改场景、改条件，确保你理解的是模式而非具体答案。

**第四步：间隔重复**
今天练完，明天再练一遍，三天后再练一遍。大脑需要时间来巩固神经通路。

### 1.5 面试高频考点

**面试官追问方式**（最容易露馅的）：
- "你说你用过窗口函数，那你说说gap-and-island怎么实现？"
- "你们公司的AB实验怎么做的？样本量怎么定的？"
- "你们用什么归因模型？为什么选这个？没考虑过别的吗？"
- "你用过DID吗？怎么检验平行趋势假设的？"
- "你写的SQL大概处理多少数据？做过哪些优化？"

## 🧩 区块2：知识点详细拆解

### 2.1 薄弱点1：窗口函数——从"知道语法"到"30秒内判断该用哪个函数"

**为什么重要**：窗口函数是SQL面试中占比最高(30%+)的知识点，也是实际工作中每天都会用到的工具。但大多数人只会ROW_NUMBER。

**核心映射表——需求→函数**：

| 需求 | 该用什么 | 关键特征 | 面试常考 |
|------|---------|----------|----------|
| "每个品类销售额前3名" | ROW_NUMBER + PARTITION BY | 分组+排名，不管并列 | ✅ 必考 |
| "并列排名（排行榜）" | RANK | 并列跳号，第1,1,3 | ✅ 必考 |
| "等级划分（不跳号）" | DENSE_RANK | 并列不跳号，第1,1,2 | ✅ 偶尔考 |
| "本月和上月的对比" | LAG(column, 1) | 环比，拿前一行 | ✅ 必考 |
| "去年同期对比" | LAG(column, 12) | 同比，拿前12行(月) | ✅ 常考 |
| "累计销售" | SUM OVER (ORDER BY) | 累积，默认到当前行 | ✅ 必考 |
| "移动平均" | AVG OVER (ROWS BETWEEN) | 窗口帧，近N行平均 | ✅ 常考 |
| "连续登录≥N天" | ROW_NUMBER + 日期减法 | gap-and-island | ✅ ✅ 最高频 |
| "首次/末次购买商品" | FIRST_VALUE / LAST_VALUE | 窗口首尾值 | ✅ 偶尔考 |
| "按百分比分桶" | NTILE(4) | 四分位数、十分位数 | ✅ 偶尔考 |

**gap-and-island三步法详述**：
```sql
-- 步骤1：去重（同一用户同一天多次登录只保留一次）
WITH dedup AS (
    SELECT DISTINCT user_id, login_date FROM logins
),
-- 步骤2：编号（每个用户按日期排序编号）
numbered AS (
    SELECT user_id, login_date,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY login_date) AS rn
    FROM dedup
),
-- 步骤3：日期减编号（连续的日期会得到相同的参考日期）
grouped AS (
    SELECT user_id, login_date,
           DATE_SUB(login_date, INTERVAL rn DAY) AS ref_date
    FROM numbered
)
-- 步骤4：按参考日期分组，统计连续天数
SELECT user_id, ref_date,
       MIN(login_date) AS start_date,
       MAX(login_date) AS end_date,
       COUNT(*) AS consecutive_days
FROM grouped
GROUP BY user_id, ref_date
HAVING COUNT(*) >= 7;
```

**为什么日期减序号能工作？—直觉解释**：
- 如果日期是连续的（1月1日, 1月2日, 1月3日），它们的序号分别是1, 2, 3
- 日期减序号：1月1日-1=12月31日, 1月2日-2=12月31日, 1月3日-3=12月31日
- 连续的日期得到相同的参考日期，中断的日期得到不同的参考日期
- 于是"连续"这个相对概念被转化为"相同"这个绝对概念——GROUP BY就能工作了

### 2.2 薄弱点2：AB实验——从"知道p<0.05"到"能独立设计和审阅实验"

**为什么重要**：AB实验是互联网公司做决策的核心工具。一个不懂AB实验的分析师会犯两种致命错误：(1)把不显著的当显著 (2)把显著的效果当业务价值。

**AB实验的五张"必会牌"**：

**牌1：样本量计算**
```python
# 给定参数计算所需样本量
from scipy import stats
import numpy as np

def sample_size_per_group(baseline_rate, mde, alpha=0.05, power=0.8):
    """
    baseline_rate: 基线转化率（如10% = 0.10）
    mde: 最小可检测效应（如2% = 0.02，即从10%到12%）
    alpha: 第一类错误率（默认0.05）
    power: 统计功效（默认0.8）
    """
    z_alpha = stats.norm.ppf(1 - alpha / 2)  # 双尾，约1.96
    z_beta = stats.norm.ppf(power)            # 约0.84
    p = baseline_rate
    delta = mde

    # 两个比例比较的样本量公式
    n = (2 * p * (1 - p) * (z_alpha + z_beta)**2) / (delta**2)
    return int(np.ceil(n))

# 示例：基线10%，MDE=2%，alpha=0.05，power=0.8
n = sample_size_per_group(0.10, 0.02)
print(f"每组需要: {n} 用户")
print(f"总计需要: {n * 2} 用户")
# 输出：每组约3500+，总计约7000+
```

**关键理解**：
- MDE越小 → 需要样本越大（检测微弱效果需要更多数据）
- 基线越接近50% → 需要样本越大（方差最大）
- Power越高 → 需要样本越大
- alpha越小 → 需要样本越大

**牌2：Peeking——为什么不能"每天看p值"？**

Peeking（偷看）指的是在实验进行过程中反复查看p值，一旦"显著"就停止实验。这个操作会让实际第一类错误率远超5%。

直觉解释：你每天抛一次硬币，如果你告诉自己"出现5次连续正面我就停"，那么最终"出现至少一次5连续正面"的概率远大于单次测试的3.125%。同理，每天做一次假设检验，只要有一次p<0.05就停——实际假阳性率可能高达20-30%。

**牌3：SRM（样本比不匹配）**
- 如果实验设计是50:50，但实际分配变成了53:47——有问题
- 可能原因：分流bug、某些用户被排除、hash算法问题
- 检验方法：卡方检验，检验实际分配是否偏离期望分配

**牌4：护栏指标**
- 主指标之外必须配1-2个护栏指标
- 护栏指标的角色：确保"主指标提升"不是以牺牲用户体验或收入质量为代价
- 典型护栏：退款率、卸载率、客服投诉量

**牌5：效应量 vs p值**
- p值告诉你"效应存在"，效应量告诉你"效应有多大"
- 大样本下微小的效应也可能p<0.05，但业务上毫无意义
- 要同时报告p值和效应量（相对提升率、Cohen's d等）

### 2.3 薄弱点3：归因模型——从"只会末次点击"到"根据场景选模型"

**为什么重要**：归因模型决定了你把功劳（和预算）分给谁。模型选错了，你可能在砍掉最重要的"播种渠道"。

**六种归因模型的适用场景矩阵**：

| 模型 | 逻辑 | 适用的决策周期 | 高估谁 | 低估谁 | 最佳场景 |
|------|------|--------------|--------|--------|----------|
| 末次点击 | 功劳全给最后触达 | 短（分钟级） | 搜索/直接 | 展示/社媒 | 外卖、打车 |
| 首次点击 | 功劳全给首次触达 | 长（月级） | 展示/社媒 | 搜索 | 品牌建设、拉新 |
| 线性 | 所有触点均分 | 中 | 无 | 无 | 触点少的场景 |
| 时间衰减 | 越靠近转化权重越大 | 中长 | 中后段触点 | 早期触点 | 内容营销 |
| 位置模型 | 首末各40%，中间20%均分 | 中 | 首末触点 | 中间触点 | B2B长链路 |
| 数据驱动 | 用算法（如Shapley值）算 | 任意 | 取决于模型 | 取决于模型 | 数据充足时 |

**选择归因模型的决策树**：
```
决策周期短（<1天）？
  ├── 是 → 末次点击基本够用
  └── 否 → 触点数量多（>5个）？
           ├── 是 → 数据驱动或时间衰减
           └── 否 → 主要目标是拉新？
                    ├── 是 → 首次点击
                    └── 否 → 线性或位置模型
```

**关键验证方法**：同一份数据，用2-3种不同归因模型计算各渠道ROI。看哪个渠道的ROI排名变化最大。如果信息流在末次点击排倒数但首次点击排第一——你现在的预算分配可能完全错了。

### 2.4 薄弱点4：因果推断——从"知道四个缩写"到"能判断何时用哪个方法"

**为什么重要**：不是所有决策都能用AB实验验证——有些策略无法随机化（如价格调整、政策变化）。因果推断就是"没有随机实验时的AB实验"。

**四种方法的适用决策矩阵**：

| 方法 | 需要的条件 | 核心前提 | 如何检验前提 | 典型场景 |
|------|-----------|---------|-------------|----------|
| DID | 有处理组和对照组，有前后两期数据 | 平行趋势：无处理时两组变化趋势相同 | 画趋势图、做安慰剂检验 | 政策评估、区域推广 |
| PSM | 有丰富的前置特征 | 条件独立：给定特征后，处理分配近乎随机 | 平衡性检验（标准化偏差<10%） | 用户行为分析、会员效果 |
| IV | 找到工具变量 | ①与处理相关 ②只通过处理影响结果 | 弱工具变量检验(F>10)、过度识别检验 | 价格弹性、广告效果 |
| RDD | 有一个明确的断点 | 断点附近处理概率跳跃、特征连续 | McCrary检验（密度连续）、伪断点检验 | 分数线、资格线政策 |

**最常混淆的地方**：
- DID要求平行趋势——如果处理前两组趋势已经不同，DID不能用
- PSM只能控制"可观测的"混淆——如果混淆变量不可观测，PSM也无能为力
- IV最难的是"找到好工具变量"——多数候选工具变量经不起检验
- RDD只在断点附近有效——远离断点的样本不在识别范围内

### 2.5 薄弱点5：SQL优化——从"SQL写对了"到"SQL写得快"

**为什么重要**：大数据量下（亿级行数），写对的SQL不一定能跑出来。性能优化能力是区分初级和高级分析师的重要标志。

**EXPLAIN输出字段的核心解读**：

| 字段 | 含义 | 好的信号 | 坏信号 |
|------|------|---------|--------|
| type | 访问类型 | const, eq_ref, ref, range | ALL, index |
| key | 实际使用的索引 | 有值（索引名） | NULL |
| rows | 预估扫描行数 | 越小越好 | 接近表行数 |
| Extra | 额外信息 | Using index（覆盖索引） | Using filesort, Using temporary |

**索引失效五大场景**：
1. `LIKE '%xxx'` —— 前导百分号导致索引失效
2. 在索引列上使用函数 —— `WHERE YEAR(date) = 2024`
3. 类型隐式转换 —— 字符串列传数字
4. OR条件 —— 不是所有OR列都有索引
5. 不等于(!= / <>) —— 大多数情况下不走索引

**SQL优化速查清单**：
- [ ] SELECT只选需要的列（不要SELECT *）
- [ ] WHERE条件尽量精确（缩小扫描范围）
- [ ] JOIN前先过滤（减少参与连接的行数）
- [ ] 使用覆盖索引（避免回表查询）
- [ ] 分页使用游标而非OFFSET
- [ ] 避免在WHERE中使用函数
- [ ] 大表JOIN时小表驱动大表

## 💻 区块3：动手练习

### 练习1：窗口函数——不看笔记手写三道SQL

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)
conn = sqlite3.connect(':memory:')

# 建模拟数据
n = 500
df = pd.DataFrame({
    'user_id': np.random.choice(['A','B','C','D','E','F','G','H'], n),
    'order_date': np.random.choice(pd.date_range('2026-01-01','2026-03-31'), n),
    'category': np.random.choice(['服装','数码','食品','美妆'], n),
    'amount': np.round(np.random.lognormal(4, 0.7, n), 2)
})
df['order_date'] = df['order_date'].astype(str)
df.to_sql('orders', conn, index=False)

print("练习1.1：每个品类消费金额排名（用ROW_NUMBER和RANK各写一遍，对比并列值差异）")
q1 = """
SELECT category, user_id,
       SUM(amount) AS total,
       ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rn,
       RANK() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS rnk,
       DENSE_RANK() OVER(PARTITION BY category ORDER BY SUM(amount) DESC) AS drnk
FROM orders
GROUP BY category, user_id
ORDER BY category, rn
LIMIT 20
"""
print(pd.read_sql(q1, conn))

print("\n练习1.2：每个用户累计消费（按日期排序）+ 3单移动平均")
q2 = """
SELECT user_id, order_date, amount,
       SUM(amount) OVER(PARTITION BY user_id ORDER BY order_date
                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative,
       AVG(amount) OVER(PARTITION BY user_id ORDER BY order_date
                        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3
FROM orders
ORDER BY user_id, order_date
LIMIT 20
"""
print(pd.read_sql(q2, conn))

print("\n练习1.3：每个用户消费金额的环比变化率（LAG）")
q3 = """
SELECT user_id, order_date, amount,
       LAG(amount, 1) OVER(PARTITION BY user_id ORDER BY order_date) AS prev_amount,
       ROUND((amount - LAG(amount, 1) OVER(PARTITION BY user_id ORDER BY order_date)) /
             LAG(amount, 1) OVER(PARTITION BY user_id ORDER BY order_date) * 100, 1) AS mom_pct
FROM orders
ORDER BY user_id, order_date
LIMIT 20
"""
print(pd.read_sql(q3, conn))

conn.close()
print("\n✅ 如果三道SQL都能独立写出来（不看笔记）→ 窗口函数过关")
print("⚠ 如果卡在某道题上 → 回到W5D1-D2重新练习")

### 练习2：AB实验——识别实验报告中的全部问题

In [ ]:
print("=" * 60)
print("审阅以下AB实验报告摘要，找出所有问题")
print("=" * 60)
print()

report = """
【实验报告】测试新版首页UI

实验设计：50%用户看新版，50%看旧版
跑了7天，产品经理每天看一次p值。
前4天p值在0.1-0.3之间波动，第5天p=0.04→决定停止实验并宣布新版有效。

结果：
- 主指标"下单转化率"：从10.2%→10.5%，p=0.08（不显著）
- "首页点击率"：从35%→42%，p=0.01（显著！）
- "分享按钮点击率"：从2.1%→2.8%，p=0.03（显著！）
- "退款率"：未查看

结论：新版首页有效（首页点击率和分享率显著提升），建议全量。
"""
print(report)
print()

issues_found = [
    ("问题1: Peeking",
     "每天看p值，看到显著就停——实际第一类错误率远超5%",
     "正确做法：提前确定样本量和跑量时间，到时间一次性分析",
     "来源：W7D5"),
    ("问题2: 主指标偷换(HARKing)",
     "主指标(下单转化率)p=0.08不显著，转而强调次要指标",
     "正确做法：预注册主指标，以主指标为准",
     "来源：W7D5"),
    ("问题3: 多重检验未校正",
     "至少看了3个指标，2个'显著'但未做校正",
     "Bonferroni校正后阈值=0.05/3=0.017，分享率p=0.03也不显著了",
     "来源：W8D5"),
    ("问题4: 缺少护栏指标",
     "点击率涨了但不知道退款率是否恶化——可能用户在'被诱导点击'",
     "正确做法：至少配置退款率和卸载率作为护栏",
     "来源：W7D3"),
    ("问题5: 未报告效应量",
     "7pp点击率提升（相对+20%）看起来很大，但主指标0.3pp（相对+3%）可能无业务意义",
     "正确做法：同时报告p值和效应量(相对提升%或Cohen's d)",
     "来源：W7D2"),
    ("问题6: 未检查SRM",
     "没有验证两组人数是否真的是50:50——如果是53:47，所有结论都有问题",
     "正确做法：实验开始后24小时内做SRM检查",
     "来源：W8D2"),
    ("问题7: 提前停止未用序贯检验",
     "即使需要提前停止，也应该用序贯检验(sequential testing)而非反复做固定样本检验",
     "固定样本检验+v.s.早停=假阳性率失控",
     "来源：W7D5"),
]

for title, problem, solution, source in issues_found:
    print(f"🔴 {title}")
    print(f"   问题：{problem}")
    print(f"   方案：{solution}")
    print(f"   ({source})")
    print()

print("=" * 60)
print("自评：")
print("  找到6-7个问题 → AB实验功底扎实 ✅")
print("  找到4-5个问题 → 基本过关，但还有盲点 📌")
print("  找到2-3个问题 → 回去复习W7D5(Peeking)和W8D5(避坑) ⚠")
print("  找到<2个问题  → 重新学一遍W7-W8 🔴")

### 练习3：归因模型——同一份数据，三种模型下的ROI排名差异

In [ ]:
import numpy as np
import pandas as pd

# 模拟6个渠道的触点数据（预算、收入、触点次数分布）
channels = ['信息流广告', '社媒KOL', '搜索广告', '邮件营销', '直接访问', '联盟推广']
budget = np.array([80, 30, 50, 15, 25, 20])  # 万元
revenue = np.array([200, 120, 175, 45, 80, 55])  # 万元

# 三种归因模型的权重（基于触点在用户旅程中的位置）
# 信息流通常在首次，搜索在末次，直接访问在末次
last_click = np.array([0.05, 0.10, 0.35, 0.05, 0.35, 0.10])
first_click = np.array([0.45, 0.20, 0.10, 0.05, 0.10, 0.10])
linear = np.array([0.167, 0.167, 0.167, 0.167, 0.167, 0.167])
time_decay = np.array([0.08, 0.12, 0.30, 0.20, 0.25, 0.05])

print("同一份数据，四种归因模型下的渠道ROI：")
print("=" * 85)
print(f"{'渠道':<12} {'预算':>6} {'收入':>6} {'末次ROI':>8} {'首次ROI':>8} {'线性ROI':>8} {'衰减ROI':>8}")
print("-" * 85)

for i, ch in enumerate(channels):
    roi_last = revenue[i] * last_click[i] / budget[i]
    roi_first = revenue[i] * first_click[i] / budget[i]
    roi_linear = revenue[i] * linear[i] / budget[i]
    roi_decay = revenue[i] * time_decay[i] / budget[i]
    print(f"{ch:<12} {budget[i]:>6} {revenue[i]:>6} {roi_last:>8.1f} {roi_first:>8.1f} {roi_linear:>8.1f} {roi_decay:>8.1f}")

print()

# 排名变化分析
last_rank = np.argsort([-revenue[i]*last_click[i]/budget[i] for i in range(6)])
first_rank = np.argsort([-revenue[i]*first_click[i]/budget[i] for i in range(6)])

print("关键发现：")
print("  1. 末次点击→搜索和直接访问ROI最高（收割渠道）→ 你会给它们最多预算")
print("  2. 首次点击→信息流ROI最高（播种渠道）→ 但末次模型下它被严重低估")
print("  3. 如果只用末次点击，你会砍掉信息流预算——然后新用户来源就断了")
print("  4. 线性模型→所有渠道均分，最'公平'但也最不精确")
print("  5. 时间衰减→中间偏后的渠道权重更高，兼顾播种和收割")
print()

# 排名变化矩阵
print("排名变化矩阵：")
df_rank = pd.DataFrame({
    '渠道': channels,
    '末次点击排名': [f'第{x+1}名' for x in last_rank],
    '首次点击排名': [f'第{x+1}名' for x in first_rank],
})
print(df_rank.to_string(index=False))
print()
print("→ 如果同一个渠道在两种模型中排名差距>2名，你的归因模型选择就非常重要")

### 练习4：因果推断——方法选择判断

In [ ]:
print("=" * 60)
print("场景判断：以下场景该用哪种因果推断方法？")
print("=" * 60)
print()

scenarios = [
    {
        "场景": "某市2024年7月提高了最低工资标准，你想评估对就业的影响",
        "数据": "有该市和周边城市2023-2025年的月度就业数据",
        "推荐": "DID",
        "原因": "有处理组(该市)和对照组(周边城市)，有前后数据，可以检验平行趋势"
    },
    {
        "场景": "你想评估"成为会员"对用户消费金额的影响",
        "数据": "有所有用户的人口统计、行为特征、消费数据，但不是随机分配会员",
        "推荐": "PSM",
        "原因": "有丰富的前置特征可以建模成为会员的概率，在倾向得分匹配后比较"
    },
    {
        "场景": "你想评估"广告曝光"对购买概率的影响",
        "数据": "有广告竞价中的随机扰动(有些人因为出价略高而看到广告)",
        "推荐": "IV",
        "原因": "广告竞价中的随机扰动可以作为工具变量——与曝光相关，但不直接影响购买"
    },
    {
        "场景": "你想评估"考上重点高中"对高考成绩的影响",
        "数据": "有考生成绩数据，重点高中有明确的分数线(600分)",
        "推荐": "RDD",
        "原因": "分数线是一个明确的断点——599分和601分的学生在能力上几乎相同"
    },
    {
        "场景": "你想评估"新功能上线"对用户活跃度的影响——可以随机分组",
        "数据": "可以随机分配50%用户看到新功能",
        "推荐": "AB实验",
        "原因": "有随机化能力的时候，直接做AB实验是最干净的方法——不需要因果推断"
    },
]

for i, s in enumerate(scenarios, 1):
    print(f"场景{i}：{s['场景']}")
    print(f"  数据：{s['数据']}")
    print(f"  ✅ 推荐：{s['推荐']}")
    print(f"  原因：{s['原因']}")
    print()

print("🎯 关键原则：")
print("  1. 能随机化→AB实验（最干净）")
print("  2. 有前后+有对照组→DID（最直观）")
print("  3. 有丰富特征→PSM（最常用）")
print("  4. 有工具变量→IV（最精巧，也最难找到好IV）")
print("  5. 有明显断点→RDD（断点附近最可信）")

## 📊 区块4：业务场景应用

### 场景1：你的同事做了一个"显著"的实验，但你不放心

**背景**：同事小王兴冲冲地在群里分享："我的新推送文案实验显著了！p=0.03，点击率从5%涨到6.2%！建议全量！"

你作为团队里的分析负责人，觉得哪里不对劲。打开实验后台发现：
1. 实验跑了12天，但小王在第6天就看到p=0.04，停了实验（实际上数据还在积累，他又"回顾"了一次）
2. 实验共看了5个指标，只报告了"点击率"这一个显著的
3. 主指标其实是"下单转化率"，p=0.15——但小王没提
4. "卸载率"这个护栏指标没配置

**你的处理方式**：

你跟小王说：
- "你这个实验Peeking了——第6天就停了，α被放大了。实际假阳性率可能到20-30%。"
- "5个指标里挑1个显著的——这大概率是假阳性。用Bonferroni校正，p值阈值应该是0.05/5=0.01，你的0.03不够。"
- "主指标不显著，你换到次要指标——这是HARKing(Hypothesizing After Results are Known)。"
- "更重要的是，你不知道推送文案是否让用户反感——卸载率没看。"

小王愣住了："那……我的实验白做了？"

你说："不是白做。数据告诉我们新文案的点击率确实可能更高——但需要重新设计实验：预注册主指标、定好样本量和停止规则、配护栏指标、一次性分析。如果重新做后主指标(p<0.05)且护栏不恶化——那才是真有效。"

**关键启示**：作为分析师，你的价值不仅是自己做对的分析，还包括**识别团队里其他人的分析错误**。一个团队里只要有一个人懂Peeking和多重检验，就能避免整个团队被假阳性结果带偏。

### 场景2：归因模型导致预算误判

**背景**：市场部准备砍掉信息流广告投放——因为末次点击归因显示信息流ROI只有0.7，而搜索广告ROI高达2.1。

**你怎么分析**：

1. 先理解用户旅程：买一件3000元家电的典型路径是——
   - 第1天：刷信息流看到品牌广告（首次触达）
   - 第3天：小红书看到KOL推荐
   - 第7天：搜索品牌名+型号
   - 第8天：点击搜索广告下单（末次触达）

2. 用首次点击模型重算：信息流ROI从0.7→3.5，原来是"播种英雄"被"收割模型"误判了。

3. 你的建议：信息流不砍，但要优化投放人群和素材质量。同时引入时间衰减归因模型，让两个渠道都得到合理的功劳分配。

**关键启示**：归因模型的选择决定了预算分配。一个错误的模型会让你的获客管道逐渐干涸。

### 场景3：SQL优化——从30分钟到30秒

**背景**：财务的月度报表查询要跑30分钟，每次月度汇报前都要提前半天跑数据。

**你怎么优化**：
1. 先EXPLAIN看执行计划 → type=ALL（全表扫描），rows=5000万
2. 在WHERE条件列上建了复合索引 → type变成range，rows降到50万
3. 改写SQL：先聚合小表再JOIN大表 → 避免了全表笛卡尔积
4. 最终：30分钟 → 30秒，快了60倍

**关键启示**：SQL优化不是玄学，是有方法论的系统工程。从EXPLAIN开始，从索引开始。

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 不看笔记能写出ROW_NUMBER、RANK、DENSE_RANK、LAG、SUM OVER的SQL
- [ ] 能说出AB实验报告中至少3个常见的"坑"（Peeking、多重检验、主指标偷换）
- [ ] 能解释为什么"末次点击归因"会高估收割渠道
- [ ] 能说出至少3种索引失效的场景
- [ ] 能说出DID/PSM/IV/RDD的中文全称和一句话概括

**答案提示**：
- 5种窗口函数：ROW_NUMBER(严格唯一)、RANK(跳跃)、DENSE_RANK(连续)、LAG(向前取)、SUM OVER(累计)
- 3个坑：Peeking→α膨胀、多重检验→假阳性增加、HARKing→主指标不显著换次要指标
- 末次点击高估收割渠道(搜索/直接)因为它们总是在最后出现
- 索引失效：LIKE '%x'、YEAR(date)、类型不匹配、OR、!=
- DID=双重差分、PSM=倾向得分匹配、IV=工具变量、RDD=断点回归

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能独立写出gap-and-island的SQL（找连续登录≥N天）
- [ ] 拿到一份实验报告，能在5分钟内找出分析方法上的所有问题
- [ ] 同一份数据用2种归因模型计算，能解释结果差异的业务含义
- [ ] 能用EXPLAIN分析慢查询并给出优化方案
- [ ] 能说出每种因果推断方法的前提假设

**答案提示**：
- gap-and-island：去重→ROW_NUMBER→日期减序号→GROUP BY
- 实验报告：Peeking/Multiple testing/HARKing/SRM/护栏/效应量——六张牌一张张检查
- 归因差异：末次高估收割(ROI高)，首次高估播种(ROI高)
- EXPLAIN：看type(最好const/eq_ref/ref，最差ALL)、key(是否用索引)、rows(扫描行数)
- 前提：DID=平行趋势、PSM=条件独立、IV=相关+排他、RDD=局部随机化

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能设计一个完整的AB实验方案（含样本量计算、SRM、护栏、停止规则、效应量）
- [ ] 能在团队中建立"实验质量审查"的checklist
- [ ] 能用数据说服团队从末次点击归因切换到更合理的模型
- [ ] 能设计包含分区、索引、查询重写的完整SQL优化方案
- [ ] 能在面试中对"你用过因果推断吗？"做出一段结构化的回答

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W1-W11全部内容 | 今天聚焦的模块来自前面每天的学习 |
| **← 昨日** | 盲区排查 | 今天具体攻克哪些模块直接来自昨天的自评结果 |
| **→ 明日** | 针对性补漏(下)——分析表达 | 技术补完后，表达力决定你的分析价值能传递多少 |
| **→ 周四** | 综合模拟场景 | 今天补的技术明天就要用在实际场景中 |
| **→ W5** | 窗口函数原始内容 | 如果今天还卡→回W5D2重新练gap-and-island |
| **→ W7-W8** | AB实验原始内容 | 如果今天还卡→回W7D2(样本量)和W7D5(Peeking) |
| **→ W9** | 因果推断原始内容 | 如果今天还卡→回W9D1-D4逐个方法回顾 |

### 横向连接：五个薄弱模块之间的协同关系

这五个模块不是独立的——它们在真实项目中经常一起出现：
- **AB实验 + SQL**：实验数据的提取和分析需要写大量SQL
- **AB实验 + 因果推断**：当无法随机化时，因果推断是AB实验的替代方案
- **归因模型 + SQL**：归因模型的计算（触点提取、权重分配）依赖复杂SQL
- **SQL优化 + 归因模型**：大数据量下的归因计算需要优化后的SQL
- **因果推断 + 归因模型**：归因本质上是因果问题——哪个渠道导致了转化？

## 📝 区块7：当日小结

> **针对性补漏的本质不是"重新学一遍"，而是"找到具体卡点→用刻意练习打通→检验能独立输出"。五大常见薄弱模块：窗口函数（需求→函数映射表）、AB实验（五张必会牌：样本量/Peeking/SRM/护栏/效应量）、归因模型（六种模型的适用矩阵+决策树）、因果推断（四种方法的前提假设+检验方法）、SQL优化（EXPLAIN解读+索引失效五大场景）。补漏四步法：找到具体卡点→获取正确模式→变式练习→间隔重复。**

**三条金句带回家**：
1. 补漏不是"重新学"，是"找到卡点→刻意练习→能独立输出"
2. AB实验的五张牌（样本量/Peeking/SRM/护栏/效应量）缺一不可
3. 归因模型选错了=预算分配错了=你在花钱毁灭自己的增长管道

### 🔖 课后拓展

1. **练习**：关掉笔记，独立写出gap-and-island的SQL（连续登录≥7天）。如果30分钟内写不出来→今天最重要的事就是这个。

2. **思考**：如果公司的AB实验平台不支持CUPED和序贯检验，你怎么在分析层面补救？

3. **预习**：明天（周三）补"分析表达"——技术能力过关后，表达力决定你的分析有多少能落地。

4. **面试准备**：背诵以下面试题答案：
   - "你用过哪些窗口函数？说一个你用的最熟的应用场景"
   - "你们公司的AB实验怎么做的？你怎么保证实验质量？"
   - "除了末次点击归因，你还用过什么归因模型？为什么选那个？"